In [1]:
import os
import xml.etree.ElementTree as ET

#### VOC to MOT——gt

In [8]:
dataset_dir = "D:\Dataset\SFIST"
# video_types = ['train', 'test']
video_types = ["test"]

for video_type in video_types:
    video_dir = os.path.join(dataset_dir, video_type, "video")
    video_list = os.listdir(video_dir)

    for video in video_list:
        if video != "03.mp4":
            continue

        gt_dir = os.path.join(dataset_dir, video_type, video.split(".")[0], "gt")
        if not os.path.exists(gt_dir):
            os.makedirs(gt_dir)
        label_dir = os.path.join(dataset_dir, video_type, "label", video.split(".")[0])
        label_num = len(os.listdir(label_dir))
        label_list = [
            os.path.join(label_dir, str(i) + ".xml") for i in range(label_num)
        ]

        classes = []

        if video in ["04.mp4"]:
            offset_ratio = 1920 / 1440
        else:
            offset_ratio = 1

        for idx, label in enumerate(label_list):

            if not os.path.exists(label):
                continue

            f = open(label)
            xml_text = f.read()
            root = ET.fromstring(xml_text)
            f.close()
            for obj in root.iter("object"):
                cls = obj.find("name").text
                if cls not in classes:
                    classes.append(cls)
        with open(os.path.join(gt_dir, "gt.txt"), "a") as out:
            for cls in classes:
                for idx, label in enumerate(label_list):
                    # if cls == '7' and idx == 752:
                    #     print('1')
                    out_text = str(idx + 1) + ","

                    if not os.path.exists(label):
                        continue

                    f = open(label)
                    xml_text = f.read()
                    root = ET.fromstring(xml_text)
                    f.close()

                    for obj in root.iter("object"):
                        if cls in obj.find("name").text:
                            out_text += str(cls) + ","

                            xmlbox = obj.find("bndbox")
                            out_text += (
                                str(round(int(xmlbox.find("xmin").text) * offset_ratio))
                                + ","
                            )  # xmin
                            out_text += str(int(xmlbox.find("ymin").text)) + ","  # ymin
                            out_text += (
                                str(
                                    round(
                                        int(xmlbox.find("xmax").text) * offset_ratio
                                        - int(xmlbox.find("xmin").text) * offset_ratio
                                    )
                                )
                                + ","
                            )  # w
                            out_text += (
                                str(
                                    int(xmlbox.find("ymax").text)
                                    - int(xmlbox.find("ymin").text)
                                )
                                + ","
                            )  # h
                            out_text += "1,1,1\n"

                            out.write(out_text)

#### VOC to MOT——det

In [ ]:
dataset_dir = "D:\Dataset\SFIST"
video_types = ["train", "test"]

for video_type in video_types:
    video_dir = os.path.join(dataset_dir, video_type, "video")
    video_list = os.listdir(video_dir)

    for video in video_list:
        gt_dir = os.path.join(dataset_dir, video_type, video.split(".")[0], "det")
        if not os.path.exists(gt_dir):
            os.makedirs(gt_dir)
        label_dir = os.path.join(dataset_dir, video_type, "label", video.split(".")[0])
        label_num = len(os.listdir(label_dir))
        label_list = [
            os.path.join(label_dir, str(i) + ".xml") for i in range(label_num)
        ]

        classes = []

        for idx, label in enumerate(label_list):
            if not os.path.exists(label):
                continue
            f = open(label)
            xml_text = f.read()
            root = ET.fromstring(xml_text)
            f.close()
            for obj in root.iter("object"):
                cls = obj.find("name").text
                if cls not in classes:
                    classes.append(cls)
        with open(os.path.join(gt_dir, "det.txt"), "a") as out:
            for idx, label in enumerate(label_list):
                if not os.path.exists(label):
                    continue
                f = open(label)
                xml_text = f.read()
                root = ET.fromstring(xml_text)
                f.close()

                for obj in root.iter("object"):
                    out_text = str(idx + 1) + ","
                    out_text += "-1,"

                    xmlbox = obj.find("bndbox")
                    out_text += str(xmlbox.find("xmin").text) + ","  # xmin
                    out_text += str(xmlbox.find("ymin").text) + ","  # ymin
                    out_text += (
                        str(
                            int(xmlbox.find("xmax").text)
                            - int(xmlbox.find("xmin").text)
                        )
                        + ","
                    )  # w
                    out_text += (
                        str(
                            int(xmlbox.find("ymax").text)
                            - int(xmlbox.find("ymin").text)
                        )
                        + ","
                    )  # h
                    out_text += "1,-1,-1,-1\n"

                    out.write(out_text)